[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_06/1_algorithms.ipynb)

# Day 6: Algorithms (easy)

**How to work:** Name the pattern before you code. Stuck 15 min: open Hint 1. Stuck 25 min: Hint 2, then the solution. Say the time and space complexity out loud. Re-type the solution from memory if you needed it.

**Today (47 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Paint bucket | easy | 15 |
| Q2 | Where would it go? | easy | 12 |
| Q3 | First letter that appears once | easy | 10 |
| Q4 | Number that reads the same both ways (bonus) | easy | 10 |

**Handbook:** [Algorithms](https://mahsa-agz.github.io/ds-handbook/#algo), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import copy

def check(fn, cases, key=None):
    """Runs fn on each (args, expected) case and prints PASS or FAIL. key: normalise outputs (e.g. sorted)."""
    ok = 0
    for args, want in cases:
        args = args if isinstance(args, tuple) else (args,)
        try:
            got = fn(*copy.deepcopy(args))
        except Exception as e:
            got = "error: " + repr(e)
        good = (key(got) == key(want)) if (key and not isinstance(got, str)) else got == want
        ok += good
        shown = ", ".join(repr(a) for a in args)
        print(("PASS " if good else "FAIL ") + f"{fn.__name__}({shown}) -> {got!r}" + ("" if good else f"   expected {want!r}"))
    print(f"{ok}/{len(cases)} passed")

---
### Q1. Paint bucket

*easy, about 15 min*

`image` is a grid of integers (colours). Starting at cell `(sr, sc)`, repaint that cell and every cell
connected to it through up/down/left/right neighbours **of the same original colour** with `color`. Return the
grid. Diagonal cells are not connected.

Example:
```
image = [[1, 1, 1],        sr = 1, sc = 1, color = 2     result: [[2, 2, 2],
         [1, 1, 0],                                                [2, 2, 0],
         [1, 0, 1]]                                                [2, 0, 1]]
```
The bottom-right `1` stays: it only touches the start region diagonally.

*Source: [LeetCode 733](https://leetcode.com/problems/flood-fill/)*

In [ ]:
def flood_fill(image, sr, sc, color):
    pass

# tests
check(flood_fill, [
    (([[1, 1, 1], [1, 1, 0], [1, 0, 1]], 1, 1, 2), [[2, 2, 2], [2, 2, 0], [2, 0, 1]]),
    (([[0, 0, 0], [0, 0, 0]], 0, 0, 0), [[0, 0, 0], [0, 0, 0]]),
    (([[5]], 0, 0, 9), [[9]]),
    (([[1, 0], [0, 1]], 0, 0, 3), [[3, 0], [0, 1]]),
    (([[0, 0, 0], [0, 1, 1]], 1, 1, 1), [[0, 0, 0], [0, 1, 1]]),
    (([[2, 2, 3], [3, 2, 3], [2, 2, 2]], 2, 0, 7), [[7, 7, 3], [3, 7, 3], [7, 7, 7]]),
])

<details><summary><b>Hint 1</b></summary>

Signal: spread from a cell to its matching neighbours on a grid. Pattern: **graph traversal on a grid (DFS or BFS)**.

</details>

<details><summary><b>Hint 2</b></summary>

1. `old = image[sr][sc]`. If `old == color`, return the image unchanged (otherwise you loop forever).
2. Push `(sr, sc)` on a stack. Pop a cell; if it is inside the grid and has colour `old`, paint it and push its 4
   neighbours.
3. Return the image.

</details>

<details><summary><b>Solution</b></summary>

```python
def flood_fill(image, sr, sc, color):
    old = image[sr][sc]
    if old == color:
        return image
    rows, cols = len(image), len(image[0])
    stack = [(sr, sc)]
    while stack:
        r, c = stack.pop()
        if 0 <= r < rows and 0 <= c < cols and image[r][c] == old:
            image[r][c] = color                    # painting also marks it visited
            stack += [(r + 1, c), (r - 1, c), (r, c + 1), (r, c - 1)]
    return image
```

**Why:** The grid is a graph: cells are nodes, same-colour neighbours are edges. DFS from the start visits the whole connected region once. Repainting a cell doubles as the "visited" mark, so no extra set is needed, which is exactly why the `old == color` case must return early.

**Complexity:** O(rows * cols) time and space in the worst case (the stack).

**Common mistakes:** Missing the `old == color` early return (infinite loop). Checking bounds after indexing. Including diagonal neighbours. Deep recursion on a big grid can hit Python's recursion limit; the explicit stack avoids it.

**Learn more:** [algo-graphs-bfs-dfs](https://mahsa-agz.github.io/ds-handbook/#algo-graphs-bfs-dfs)

</details>

---
### Q2. Where would it go?

*easy, about 12 min*

`nums` is sorted in increasing order with distinct values. Return the index of `target` if it is present;
otherwise return the index where it would be inserted to keep the list sorted. Required: O(log n).

Examples:
- `nums = [1, 3, 5, 6], target = 5` -> `2`
- `nums = [1, 3, 5, 6], target = 2` -> `1`
- `nums = [1, 3, 5, 6], target = 7` -> `4` (after the last element)

*Source: [LeetCode 35](https://leetcode.com/problems/search-insert-position/)*

In [ ]:
def search_insert(nums, target):
    pass

# tests
check(search_insert, [
    (([1, 3, 5, 6], 5), 2),
    (([1, 3, 5, 6], 2), 1),
    (([1, 3, 5, 6], 7), 4),
    (([1, 3, 5, 6], 0), 0),
    (([1], 1), 0),
    (([1, 3], 2), 1),
    (([], 4), 0),
])

<details><summary><b>Hint 1</b></summary>

Signal: sorted input, O(log n), and the answer is a position (the first index with value >= target). Pattern: **binary search (lower bound)**.

</details>

<details><summary><b>Hint 2</b></summary>

1. Half-open range: `lo, hi = 0, len(nums)`.
2. While `lo < hi`: `mid = (lo + hi) // 2`. If `nums[mid] < target`: `lo = mid + 1`, else `hi = mid`.
3. Return `lo`: the first index whose value is >= target (it is `len(nums)` if none is).

</details>

<details><summary><b>Solution</b></summary>

```python
def search_insert(nums, target):
    lo, hi = 0, len(nums)
    while lo < hi:
        mid = (lo + hi) // 2
        if nums[mid] < target:
            lo = mid + 1
        else:
            hi = mid
    return lo

# the standard library does exactly this:
import bisect
print(bisect.bisect_left([1, 3, 5, 6], 2))   # 1
```

**Why:** "First index with value >= target" is a yes/no question that is False then True along the sorted list, so binary search can find the boundary. The same template answers found and not found in one go. Knowing `bisect_left` is a plus in exams; still be ready to write the loop.

**Complexity:** O(log n) time, O(1) space.

**Common mistakes:** Starting `hi` at `len(nums) - 1`, which can never return `len(nums)` (fails for target 7). Mixing the inclusive and half-open templates (`hi = mid - 1` with `lo < hi`).

**Learn more:** [algo-binary-search](https://mahsa-agz.github.io/ds-handbook/#algo-binary-search)

</details>

---
### Q3. First letter that appears once

*easy, about 10 min*

Return the index of the first character in `s` that occurs exactly once in the whole string, or `-1` if
every character repeats.

Examples:
- `"leetcode"` -> `0` (`l`)
- `"loveleetcode"` -> `2` (`v`)
- `"aabb"` -> `-1`

*Source: [LeetCode 387](https://leetcode.com/problems/first-unique-character-in-a-string/)*

In [ ]:
def first_uniq_char(s):
    pass

# tests
check(first_uniq_char, [
    ("leetcode", 0),
    ("loveleetcode", 2),
    ("aabb", -1),
    ("z", 0),
    ("", -1),
    ("dddccdbba", 8),
])

<details><summary><b>Hint 1</b></summary>

Signal: you need the total count of each character before you can judge the first one. Pattern: **hashing** (count, then a second pass).

</details>

<details><summary><b>Hint 2</b></summary>

1. Count every character: `Counter(s)`.
2. Walk `s` again with indexes; return the first index whose count is 1.
3. Return -1 if none.

</details>

<details><summary><b>Solution</b></summary>

```python
from collections import Counter

def first_uniq_char(s):
    count = Counter(s)
    for i, ch in enumerate(s):
        if count[ch] == 1:
            return i
    return -1
```

**Why:** Whether a character is unique depends on the whole string, so one pass to count is needed first. The second pass keeps the original order, so the first hit is the answer. Two O(n) passes are still O(n).

**Complexity:** O(n) time, O(k) space with k distinct characters (O(1) for 26 letters).

**Common mistakes:** Returning the character instead of the index. Using `s.count(ch)` inside the loop (O(n^2)). Walking the dict instead of the string (dict order is first-insertion order, which happens to work in Python 3.7+, but say why if you rely on it).

**Learn more:** [algo-hashing](https://mahsa-agz.github.io/ds-handbook/#algo-hashing)

</details>

---
### Q4. Number that reads the same both ways (bonus)

*easy, about 10 min*

Optional. Return `True` if the integer `x` reads the same left to right and right to left. Do it without
converting the number to a string.

Examples:
- `121` -> `True`
- `-121` -> `False` (reads `121-` backwards)
- `10` -> `False` (reads `01`)

*Source: [LeetCode 9](https://leetcode.com/problems/palindrome-number/)*

In [ ]:
def is_palindrome_number(x):
    pass

# tests
check(is_palindrome_number, [
    (121, True),
    (-121, False),
    (10, False),
    (0, True),
    (7, True),
    (12321, True),
    (1000021, False),
    (1221, True),
])

<details><summary><b>Hint 1</b></summary>

Signal: digits can be peeled off with `% 10` and `// 10`. Pattern: **digit math** (reverse half of the number), from the arrays and strings toolkit.

</details>

<details><summary><b>Hint 2</b></summary>

1. Negative numbers, and numbers that end in 0 (except 0 itself), are never palindromes.
2. Build `rev` from the last digits: `rev = rev * 10 + x % 10`, `x //= 10`, while `x > rev`.
3. Even digit count: `x == rev`. Odd digit count: `x == rev // 10` (drop the middle digit).

</details>

<details><summary><b>Solution</b></summary>

```python
def is_palindrome_number(x):
    if x < 0 or (x % 10 == 0 and x != 0):
        return False
    rev = 0
    while x > rev:
        rev = rev * 10 + x % 10
        x //= 10
    return x == rev or x == rev // 10
```

**Why:** Reversing only half of the digits avoids building the full reversed number (which can overflow in fixed-size integer languages) and stops early. When `x <= rev`, you have reached the middle.

**Complexity:** O(d) time where d is the number of digits (that is O(log x)), O(1) space.

**Common mistakes:** Forgetting the trailing-zero case (`10` would wrongly pass the half check). Treating negative numbers as palindromes. Using `str(x)` when the examiner asked not to.

**Learn more:** [algo-arrays-strings](https://mahsa-agz.github.io/ds-handbook/#algo-arrays-strings)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [SQL](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_06/2_sql.ipynb)